# Customer Segmentation Using RFM Analysis

**Business question:** Which customers should the retail team retain, grow, or reactivate based on purchase recency, order frequency, and revenue?

This analysis creates Tableau-ready customer and segment files from completed sales transactions. K-Means is used only to group customers with similar RFM profiles; the focus is on the business actions each segment supports.


## Import Required Libraries

Import Python libraries for data manipulation, clustering, and visualization.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Set plot style for better visuals
plt.style.use('seaborn-v0_8')  # Use seaborn-v0_8 style (compatible with newer Matplotlib)

In [2]:
# Load dataset (adjust file path as needed)
data = pd.read_excel('OnlineRetail.xlsx')  # Download from UCI or Kaggle

# Display first few rows
print("First 5 rows of the dataset:")
display(data.head())

# Check dataset info
print("\nDataset Info:")
data.info()

# Check for missing values
print("\nMissing Values:")
print(data.isnull().sum())

First 5 rows of the dataset:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB

Missing Values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


## Data Cleaning

Clean the dataset by handling missing values, removing invalid data, and preparing it for RFM analysis.

In [ ]:
# Remove transactions that cannot represent a completed customer purchase.
data = data.dropna(subset=['CustomerID']).copy()

# In this dataset, invoice numbers beginning with C are cancellations.
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')].copy()

# Keep only positive sales values. This also removes returns and invalid prices.
data = data[(data['Quantity'] > 0) & (data['UnitPrice'] > 0)].copy()

# Use clear data types before calculating the sales value.
data['CustomerID'] = data['CustomerID'].astype(int)
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

print(f"Completed transaction rows: {len(data):,}")
print(f"Customers with valid purchases: {data['CustomerID'].nunique():,}")
print(f"Valid sales revenue: £{data['TotalPrice'].sum():,.2f}")


## RFM Analysis

Calculate Recency, Frequency, and Monetary metrics for each customer to prepare for segmentation.
- **Recency**: Days since last purchase.
- **Frequency**: Number of unique purchases.
- **Monetary**: Total spending.

In [ ]:
# Use the calendar day after the final completed sale as the RFM snapshot date.
# Normalising the timestamp prevents time-of-day values from changing whole-day recency.
snapshot_date = data['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)

# Frequency is the count of unique completed invoices, not the count of item rows.
rfm = data.groupby('CustomerID', as_index=False).agg(
    Recency=('InvoiceDate', lambda dates: (snapshot_date - dates.max().normalize()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
)

print(f"RFM snapshot date: {snapshot_date.date()}")
display(rfm.head())
display(rfm[['Recency', 'Frequency', 'Monetary']].describe())


## Assign RFM Scores

Assign scores (1 to 4) to Recency, Frequency, and Monetary based on quartiles to categorize customers.
- Higher Recency score for lower days (recent purchases).
- Higher Frequency and Monetary scores for higher values.

In [5]:
import pandas as pd
import numpy as np

# Recency: Quartiles, lower is better
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1], duplicates='drop')

# Frequency: Manual bins to handle skewness
rfm['F_Score'] = pd.cut(rfm['Frequency'], bins=[0, 1, 3, 10, np.inf], labels=[1, 2, 3, 4])

# Monetary: Quartiles, higher is better
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4], duplicates='drop')

# Combine scores
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Show result
print("RFM with Scores:")
display(rfm.head())

RFM with Scores:


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
CustomerID,,,,,,,
12346,326,1,77183.60,1,1,4,114
12347,2,7,4310.00,4,3,4,434
12348,75,4,1797.24,2,3,4,234
12349,19,1,1757.55,3,1,4,314
12350,310,1,334.40,1,1,2,112


## K-Means Clustering

Apply K-Means clustering to group customers into segments based on scaled RFM metrics.

In [ ]:
# Standardize RFM metrics so different units can contribute to the cluster assignment.
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# The elbow chart is a supporting check for the four business segments used here.
inertia = []
for k in range(1, 10):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    inertia.append(model.fit(rfm_scaled).inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 10), inertia, marker='o')
plt.title('Elbow Chart for Customer Groups')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.show()

# n_init=10 makes the result reproducible across scikit-learn versions.
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster_ID'] = kmeans.fit_predict(rfm_scaled)
print(f"Silhouette score: {silhouette_score(rfm_scaled, rfm['Cluster_ID']):.3f}")


## Name the Customer Segments

K-Means cluster numbers have no business meaning by themselves. The next cell calculates each cluster's average RFM profile and assigns a clear segment name from that profile:

- **Champions:** the highest average monetary value, with very recent and frequent purchases.
- **Loyal Customers:** frequent, recent purchasers with strong customer value.
- **Active Customers:** the remaining current customer base with regular but lower purchasing activity.
- **Lapsed Customers:** the longest average time since purchase and low purchasing frequency.


In [ ]:
# Calculate average RFM values before giving each cluster a business name.
cluster_summary = rfm.groupby('Cluster_ID').agg(
    Customers=('CustomerID', 'size'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
)

# Name clusters from their calculated profile, never from the arbitrary cluster number.
champions = cluster_summary['Avg_Monetary'].idxmax()
lapsed = cluster_summary['Avg_Recency'].idxmax()
remaining_clusters = cluster_summary.drop(index=[champions, lapsed])
loyal = remaining_clusters['Avg_Frequency'].idxmax()
active = remaining_clusters.drop(index=loyal).index[0]

segment_map = {
    champions: 'Champions',
    loyal: 'Loyal Customers',
    active: 'Active Customers',
    lapsed: 'Lapsed Customers'
}
cluster_map = {cluster: f'Cluster {letter}' for cluster, letter in zip(sorted(segment_map), 'ABCD')}

# Cluster is text for Tableau; Segment is the business-friendly label.
rfm['Cluster'] = rfm['Cluster_ID'].map(cluster_map)
rfm['Segment'] = rfm['Cluster_ID'].map(segment_map)

cluster_summary['Cluster'] = cluster_summary.index.map(cluster_map)
cluster_summary['Segment'] = cluster_summary.index.map(segment_map)
display(cluster_summary[['Cluster', 'Segment', 'Customers', 'Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']].round(2))


## Create Dashboard-ready Charts

The following charts use the same colors for each segment and are saved in the `images` folder for the README and portfolio dashboard.


In [ ]:
from pathlib import Path

# Keep the segment order and colours identical across every chart.
segment_order = ['Champions', 'Loyal Customers', 'Active Customers', 'Lapsed Customers']
segment_colors = {
    'Champions': '#1B9E77',
    'Loyal Customers': '#7570B3',
    'Active Customers': '#E6AB02',
    'Lapsed Customers': '#D95F02'
}
images_dir = Path('images')
images_dir.mkdir(exist_ok=True)

# Summarise the business metrics needed in Tableau and in the portfolio README.
segment_summary = rfm.groupby('Segment').agg(
    Customers=('CustomerID', 'size'),
    Total_Revenue=('Monetary', 'sum'),
    Total_Orders=('Frequency', 'sum'),
    Average_Recency=('Recency', 'mean'),
    Average_Frequency=('Frequency', 'mean'),
    Average_Monetary=('Monetary', 'mean')
)
segment_summary['Customer_Percent'] = segment_summary['Customers'] / segment_summary['Customers'].sum() * 100
segment_summary['Revenue_Percent'] = segment_summary['Total_Revenue'] / segment_summary['Total_Revenue'].sum() * 100
segment_summary['Average_Order_Value'] = segment_summary['Total_Revenue'] / segment_summary['Total_Orders']
segment_summary = segment_summary.loc[segment_order].reset_index()

# Customers by segment.
plt.figure(figsize=(9, 5))
sns.barplot(data=segment_summary, x='Segment', y='Customers', hue='Segment', palette=segment_colors, legend=False)
plt.title('Customers by Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(images_dir / 'customers_by_segment.png', dpi=180, bbox_inches='tight')
plt.show()

# Revenue share by segment.
plt.figure(figsize=(9, 5))
sns.barplot(data=segment_summary, x='Segment', y='Revenue_Percent', hue='Segment', palette=segment_colors, legend=False)
plt.title('Revenue Share by Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Revenue Share (%)')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(images_dir / 'revenue_share_by_segment.png', dpi=180, bbox_inches='tight')
plt.show()

# Recency and monetary value by segment.
plt.figure(figsize=(10, 6))
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Segment', hue_order=segment_order,
                palette=segment_colors, alpha=0.7, s=45)
plt.yscale('log')  # Keeps lower-value and higher-value customers visible together.
plt.title('Recency vs Monetary Value by Segment (Log Scale)')
plt.xlabel('Recency (days since last purchase)')
plt.ylabel('Monetary Value (£, log scale)')
plt.legend(title='Customer Segment')
plt.tight_layout()
plt.savefig(images_dir / 'recency_vs_monetary.png', dpi=180, bbox_inches='tight')
plt.show()


## Business Recommendations

- **Champions:** protect the relationship with early access, recognition, and service recovery if an issue occurs.
- **Loyal Customers:** use bundles and cross-sell offers to increase value without discounting the whole customer base.
- **Active Customers:** encourage a next purchase with timely product recommendations after a completed order.
- **Lapsed Customers:** run a measured reactivation campaign and compare response with a holdout group before expanding it.
- **All segments:** track repeat order rate, revenue, and offer response by segment in the dashboard each month.


## Export Data for Visualization
Export RFM data with clusters for Tableau/Power BI dashboards.

In [ ]:
# Export only the fields Tableau needs for its customer-level analysis.
tableau_rfm = rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Cluster', 'Segment']]
tableau_rfm.to_csv('rfm_segments_named.csv', index=False)
segment_summary.to_csv('segment_summary.csv', index=False, float_format='%.2f')

print("Saved rfm_segments_named.csv for customer-level Tableau analysis.")
print("Saved segment_summary.csv for segment-level dashboard metrics.")


## SQL Query 
Simulate a database query for RFM metrics 

In [ ]:
sql_query = """
SELECT
    CustomerID,
    DATEDIFF(DAY, MAX(CAST(InvoiceDate AS DATE)), '2011-12-10') AS Recency,
    COUNT(DISTINCT InvoiceNo) AS Frequency,
    SUM(Quantity * UnitPrice) AS Monetary
FROM retail_data
WHERE CustomerID IS NOT NULL
  AND InvoiceNo NOT LIKE 'C%'
  AND Quantity > 0
  AND UnitPrice > 0
GROUP BY CustomerID;
"""
print("SQL version of the RFM calculation:")
print(sql_query)


## Conclusion

This project turns completed retail transactions into customer segments that a marketing or retention team can use in Tableau. The main outputs are a customer-level RFM file, a segment-level business summary, and three supporting charts.

### Methodology

The analysis removes unidentified customers, cancellations, and nonpositive sales values. It calculates RFM from the day after the final completed sale, standardizes the three measures, then uses K-Means to identify four groups. The groups are named from their calculated average RFM profiles so they can support business decisions.
